# Etapa 3: Baseline e Estratégia Algorítmica

Nesta etapa, vamos criar uma simulação de **Multi-Armed Bandit**. 

**O Problema do Bandit aqui:**
Temos um orçamento limitado de mensagens/ligações. Precisamos descobrir qual **segmento de clientes** (neste exemplo, usaremos a profissão - `job`) tem a maior probabilidade de conversão. 

Cada "Braço" (Arm) do nosso Bandit será um segmento de profissão. 
- **Baseline:** Escolher segmentos aleatoriamente para enviar a mensagem (Regra Fixa/Aleatória).
- **Algoritmo Adaptativo (Thompson Sampling):** Aprender ao longo do tempo quais segmentos convertem mais e focar as mensagens neles, mas sem parar de explorar os outros de vez em quando.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from scipy.stats import beta

# Definindo semente para reprodutibilidade
np.random.seed(42)
random.seed(42)

## 1. Carregando os Dados
Para a simulação do Bandit, vamos usar a base limpa para ter acesso fácil às categorias originais.

In [ ]:
df = pd.read_csv('../data/bank_cleaned.csv')

# Nossos braços serão as categorias da coluna 'job'
arms = df['job'].unique().tolist()
print(f"Temos {len(arms)} braços (segmentos) para explorar:")
print(arms)

Vamos descobrir qual é a verdadeira taxa de conversão de cada braço apenas para termos como gabarito (o Bandit não sabe disso no início!).

In [ ]:
true_conversion_rates = df.groupby('job')['target'].mean().sort_values(ascending=False)
print("GABARITO - Taxa de Conversão Real por Segmento:")
display(true_conversion_rates)

## 2. Simulador do Ambiente
Função que simula o envio de uma mensagem para um segmento e retorna se houve conversão ou não, puxando um cliente aleatório desse segmento na base de dados.

In [ ]:
def pull_arm(arm_name):
    # Filtra os clientes que pertencem ao segmento escolhido
    segment_data = df[df['job'] == arm_name]
    # Sorteia um cliente desse segmento
    client_reward = segment_data.sample(1)['target'].values[0]
    return client_reward

## 3. Modelo Baseline (Aleatório)
Vamos simular 5.000 interações escolhendo segmentos aleatoriamente.

In [ ]:
n_iterations = 5000

baseline_rewards = []
cumulative_baseline = 0
baseline_cumulative_rewards = []

for i in range(n_iterations):
    # Escolhe um braço aleatoriamente
    chosen_arm = random.choice(arms)
    
    # Observa a recompensa
    reward = pull_arm(chosen_arm)
    
    # Registra
    cumulative_baseline += reward
    baseline_cumulative_rewards.append(cumulative_baseline)

print(f"Total de conversões (Baseline): {cumulative_baseline} em {n_iterations} tentativas")

## 4. Modelo Adaptativo (Thompson Sampling)
Agora usando inferência Bayesiana para equilibrar exploração e explotação.

In [ ]:
# Inicialização dos parâmetros Alpha e Beta (Priori) para cada braço
alphas = {arm: 1 for arm in arms}
betas = {arm: 1 for arm in arms}

ts_rewards = []
cumulative_ts = 0
ts_cumulative_rewards = []
arm_selections = {arm: 0 for arm in arms}

for i in range(n_iterations):
    # 1. Amostrar um valor da distribuição Beta para cada braço
    sampled_theta = {}
    for arm in arms:
        sampled_theta[arm] = np.random.beta(alphas[arm], betas[arm])
    
    # 2. Escolher o braço com o maior valor amostrado (Exploração vs Explotação)
    chosen_arm = max(sampled_theta, key=sampled_theta.get)
    arm_selections[chosen_arm] += 1
    
    # 3. Observar a recompensa
    reward = pull_arm(chosen_arm)
    
    # 4. Atualizar as distribuições Posteriori
    if reward == 1:
        alphas[chosen_arm] += 1
    else:
        betas[chosen_arm] += 1
        
    # Registra
    cumulative_ts += reward
    ts_cumulative_rewards.append(cumulative_ts)

print(f"Total de conversões (Thompson Sampling): {cumulative_ts} em {n_iterations} tentativas")

## 5. Comparação e Avaliação

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(baseline_cumulative_rewards, label="Baseline (Aleatório)", color='gray', linestyle='--')
plt.plot(ts_cumulative_rewards, label="Thompson Sampling (Adaptativo)", color='blue')
plt.xlabel("Iterações (Mensagens Enviadas)")
plt.ylabel("Conversões Acumuladas")
plt.title("Comparação de Desempenho: Baseline vs Thompson Sampling")
plt.legend()
plt.grid(True)
plt.show()

ganho = ((cumulative_ts - cumulative_baseline) / cumulative_baseline) * 100
print(f"O Algoritmo Adaptativo obteve {ganho:.2f}% mais conversões que o Baseline!")

In [ ]:
# Vamos ver quais braços o algoritmo mais escolheu
selections_df = pd.Series(arm_selections).sort_values(ascending=False)
print("Quantidade de vezes que cada segmento foi escolhido pelo Thompson Sampling:")
display(selections_df)

plt.figure(figsize=(10, 5))
selections_df.plot(kind='bar', color='orange')
plt.title("Seleção de Braços (Explotação) pelo Thompson Sampling")
plt.ylabel("Vezes Escolhido")
plt.show()

## 6. Registro de Experimentos com MLflow (Etapa 7)
Como boa prática de MLOps, vamos registrar os resultados dessa simulação no MLflow para que possamos auditar e comparar versões futuras.

In [9]:
import mlflow

# Configurando o experimento
mlflow.set_experiment("Campanha_Marketing_Bancario_Bandit")

with mlflow.start_run():
    # 1. Registrando Parâmetros (Hiperparâmetros do Experimento)
    mlflow.log_param("algorithm", "Thompson Sampling")
    mlflow.log_param("iterations", n_iterations)
    mlflow.log_param("arms_count", len(arms))
    
    # 2. Registrando Métricas (Resultados)
    mlflow.log_metric("baseline_conversions", cumulative_baseline)
    mlflow.log_metric("thompson_sampling_conversions", cumulative_ts)
    mlflow.log_metric("percentage_gain", ganho)
    
    print("✅ Métricas registradas com sucesso no MLflow local!")
    print("Para visualizar a interface do MLflow, abra o terminal e digite: mlflow ui")

/Users/murillo/Documents/FIAP/ML/Módulo 5/TrabalhoFinal /venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
2026/09/23 20:48:48 INFO mlflow.tracking.fluent: Experiment with name 'Campanha_Marketing_Bancario_Bandit' does not exist. Creating a new experiment.


✅ Métricas registradas com sucesso no MLflow local!
Para visualizar a interface do MLflow, abra o terminal e digite: mlflow ui
